# Goal

Продолжение/дополнение к 14-ому исследованию. Ищем лучшую структуру `info_nce_projector`.

# TARGET_NOTEBOOK_FNAME

In [31]:
TARGET_NOTEBOOK_FNAME = '18d_world_model_07.ipynb'

# GRID_SEARCH_SPACE

In [32]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [33]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.launch_goal = 'train_predictor'
    
    HP.general.comment = None
    HP.general.random_seed = random.randint(0, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.dataset.train = [
        'train_dataset_seq_eq_5_max_eq_0:101',
        'train_dataset_seq_eq_5_max_eq_0:102',
        'train_dataset_seq_eq_5_max_eq_0:103',
        'train_dataset_seq_eq_5_max_eq_0:104',
        'train_dataset_seq_eq_5_max_eq_0:105',
        'train_dataset_seq_eq_5_max_eq_0:106',
        'train_dataset_seq_eq_5_max_eq_0:107',
        'train_dataset_seq_eq_5_max_eq_0:108',
        'train_dataset_seq_eq_5_max_eq_0:109',
        'train_dataset_seq_eq_5_max_eq_0:110',
    ]
    HP.dataset.test = 'test_dataset_seq_eq_5_max_eq_0:101'

    HP.model.parent = '18d_world_model_07:8'
    HP.model.sequence_length = 4
    HP.model.actions_count = 6
    HP.model.ob_shape = (1, 178, 152)
    HP.model.d_model = 256
    HP.model.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.model.encoder = dict(
        is_trainable=HP.launch_goal in ('train_encoder', 'train_all', None),
        vision_head=dict(grid=(6,6), features_counts=(16, 32, 64, 128)),
    )
    HP.model.predictor = dict(
        is_trainable=HP.launch_goal in ('train_predictor', 'train_all', None),
        down_translator=None,
        parent=None, # do not load from HP.model.parent
    )
    HP.model.renderer = dict(
        is_trainable=HP.launch_goal in ('train_encoder', 'train_all', None),
        projector=dict(type='linear'), 
        render_engine=dict(
            type='layer', 
            layers_count=4, 
            features_counts=(96, 64, 32, 16),
        ),
    )
    HP.model.info_nce_projector = dict(
        is_trainable=HP.launch_goal in ('train_predictor', 'train_all', None),
        d_inner=HP.model.d_model,
        d_projected=optuna_trial.suggest_categorical('d_projected', [128, 192, 256]),
        parent=None, # do not load from HP.model.parent
    )
    
    HP.train.epochs_count = 100
    HP.train.batch_size = 128
    HP.train.optimizer = 'AdamW'
    HP.train.max_grad_norm = 1.0
    HP.train.learn_rate = f'const({optuna_trial.suggest_float('learn_rate', 0.0001, 0.01)})'
    HP.train.bce_loss_coef = 'const(1.0)'
    HP.train.edge_loss_coef = 'const(1.0)'
    HP.train.recon_loss_coef = 'const(0.0)'
    HP.train.pred_loss = dict(
        type='InfoNCE', 
        loss_coef='const(1.0)', 
        temperature=optuna_trial.suggest_float('info_nce_loss.temperature', 0.01, 0.5)
    )
    
    return HP

# Results


Лучший результат имеет `18d_world_model_07:185` `ssim=0.9417` с `d_projected=256`.

Это очень странно. Получается, что функцию бутылочного горлышка `info_nce_projector` толком и не выполняет.  Возникла мысль, что м.б. при 256 модель просто копирует данные из последнего латента исторических обсов? Проверил это в рамках `18d_world_model_07:193` - просто использовал последний исторический латент в качестве предиктивного (по-факту это означает что берём последний обс и обходимся с ним, как будто это тот, который мы предсказываем). Получился `ssim=0.9463`. Так что гипотеза по-ходу частично подтверждается, модель обнаруживает, что если просто скопировать, то будет то, что надо. Ну и модель, наверное, не очень далека от истины, т.к. разница в 0,13 секунд меняет последний об совсем незначительно. Т.е. мы предсказываем в большинстве случаев не изменение позиции объектов на сцене, а их вариацию: крылышки вверх или вниз, ножки игрока подогнуты или нет и т.д. 

Один из вариантов - заставить модель предсказывать более далёкое будущее, например, +1 с. Либо вернуться к предсказанию изменений в рамках небольших фокусных квадратиков. В конце концов о будущем мы тоже думаем только в рамках какого-то вопроса. Даже не +1с вперёд можно думать применительно к чему-либо. Типа, что будет с этим листочком, что будет с тенью, что будет со скамейкой. Обо всём сразу невозможно подумать - только по одному за раз. 

**Выводы**
1) без `info_nce_projector` получаются `ssim` в районе `0.77` (см. `18d_study_15a.1`). 
2) посмотреть, что теперь с `MSE` будет

# System

In [34]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [35]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18d_world_model_07.ipynb',
 'target_notebook_name': '18d_world_model_07',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18d_study_15.1/18d_study_15.1.ipynb',
 'optuna_study_name': '18d_study_15.1',
 'optuna_study_serial': '15.1',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18d_study_15.1/18d_study_15.1.optuna'}

In [36]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [37]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [38]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [39]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [40]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 100
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-09-10 09:26:40,202] A new study created in Journal with name: 18d_study_15.1


2026.09.10-09:26:40.373070     0.146 >> Model instance registered, version=113
2026.09.10-09:26:40.388026     0.007 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18d_world_model_07-launch113.ipynb"
2026.09.10-09:26:40.388549     0.007 >> 1.0 idle runners exist, submitted launch "18d_world_model_07:113"; running launches=1
2026.09.10-09:29:27.602683     0.166 >> Model instance registered, version=114
2026.09.10-09:29:27.618807     0.008 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18d_world_model_07-launch114.ipynb"
2026.09.10-09:29:27.619823     0.009 >> 1.7 idle runners exist, submitted launch "18d_world_model_07:114"; running launches=2
2026.09.10-09:29:58.489036     0.187 >> Model instance registered, version=115
2026.09.10-09:29:58.503883     0.007 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18d_world_model_07-launch115.ipynb"
2026.09.10-09:29:58.504358     0.007 >> 4.5 idle runners exist, submitted launch "18d_world_model_07:115"; running launches=3
2026

KeyboardInterrupt: 

In [44]:
# @launchit.disable
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-09-10 17:00:40,612] Using an existing study with name '18d_study_15.1' instead of creating a new one.


2026.09.10-17:00:40.614213   125.532 >> Study statistics: 
2026.09.10-17:00:40.618744     0.005 >> 	Number of finished trials: 80
2026.09.10-17:00:40.619790     0.001 >> 	Number of pruned trials: 0
2026.09.10-17:00:40.620248     0.000 >> 	Number of complete trials: 71
2026.09.10-17:00:40.620779     0.001 >> Best trial:
2026.09.10-17:00:40.621315     0.001 >> 	Value: 0.9415391990914941
2026.09.10-17:00:40.621571     0.000 >> 	Model version: n/a
2026.09.10-17:00:40.621819     0.000 >> 	Params: 
2026.09.10-17:00:40.622183     0.000 >> 		d_projected: 256
2026.09.10-17:00:40.622599     0.000 >> 		learn_rate: 0.0011980459305335825
2026.09.10-17:00:40.622850     0.000 >> 		info_nce_loss.temperature: 0.22814182811425365
